In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image
from tqdm import tqdm
import os

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
BATCH_SIZE = 128
IMG_SIZE = 32
TIMESTEPS = 300
LEARNING_RATE = 1e-3
EPOCHS = 20
OUTPUT_DIR = "diffusion_outputs"

os.makedirs(OUTPUT_DIR, exist_ok = True)

def get_data_loader():
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Lambda(lambda t: (t*2)-1)
    ])
    dataset = datasets.MNIST(root = "./data", train = True, download = True, transform = transform)
    dataloader = DataLoader(dataset, batch_size = BATCH_SIZE, shuffle = True)
    return dataloader

In [ ]:
def linear_beta_schedule(timesteps, start = 0.0001, end = 0.02):
    return torch.linspace(start, end, timesteps)

betas = linear_beta_schedule(timesteps = TIMESTEPS).to(DEVICE)

alphas = 1-betas
alphas_cumprod = torch.cumprod(alphas, axis = 0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

sqrt_recip_alphas = torch.sqrt(1.0/alphas)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

posterior_variance = betas*(1. - alphas_cumprod_prev)/(1. - alphas_cumprod)


def extract(a, t, x_shape):
    batch_size = t.shape[0]
    out = a.gather(0, t.to(a.device))
    return out.reshape(
        batch_size, *((1,) * (len(x_shape) - 1))
    )



def q_sample(x_start, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x_start)
    sqrt_alphas_cumprod_t = extract(
        sqrt_alphas_cumprod, t, x_start.shape
    )
    sqrt_one_minus_alphas_cumprod_t = extract(
        sqrt_one_minus_alphas_cumprod, t, x_start.shape
    )
    return (
        sqrt_alphas_cumprod_t * x_start
        + sqrt_one_minus_alphas_cumprod_t * noise
    )

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = torch.log(torch.tensor(10000.0, device = device))/ (half_dim-1)
        embeddings = torch.exp(torch.arange(half_dim, device = device)* -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim = -1)
        return embeddings

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, up = False):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2*in_ch, out_ch, 3, padding= 1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding= 1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding= 1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()
    
    def forward(self, x, t):
        h = self.bnorm1(self.relu(self.conv1(x)))
        time_emb = self.relu(self.time_mlp(t))
        time_emb = time_emb[(...,)+(None, )*2]
        h = h + time_emb
        h = self.bnorm2(self.relu(self.conv2(h)))
        return self.transform(h)


class SimpleUnet(nn.Module):
    def __init__(self):
        super().__init__()

        image_channels = 1
        down_channels = (64, 128, 256, 512, 1024)
        up_channels = (1024, 512, 256, 128, 64)
        time_emb_dim = 32

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )

        self.conv0 = nn.Conv2d(
            image_channels, down_channels[0], 3, padding=1
        )

        self.downs = nn.ModuleList([
            Block(
                down_channels[i],
                down_channels[i + 1],
                time_emb_dim
            )
            for i in range(len(down_channels) - 1)
        ])

        self.ups = nn.ModuleList([
            Block(
                up_channels[i],
                up_channels[i + 1],
                time_emb_dim,
                up=True
            )
            for i in range(len(up_channels) - 1)
        ])

        self.output = nn.Conv2d(
            up_channels[-1], 1, 1
        )

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x = self.conv0(x)

        residual_inputs = []

        for down in self.downs:
            x = down(x, t)
            residual_inputs.append(x)

        for up in self.ups:
            residual_x = residual_inputs.pop()
            x = torch.cat((x, residual_x), dim=1)
            x = up(x, t)

        return self.output(x)


def forward(self, x, timestep):
    t = self.time_mlp(timestep)
    x = self.conv0(x)
    residual_inputs = []
    for down in self.downs:
        x= down(x, t)
        residual_inputs.append(x)

    for up in self.ups:
        residual_x = residual_inputs.pop()
        x = torch.cat((x, residual_x), dim = 1)
        x = up(x, t)
    return self.output(x)

In [ ]:
@torch.no_grad()
def p_sample(model, x, t, t_index):
    betas_t = extract(betas, t, x.shape)
    sqrt_one_minus_alphas_cumprod_t = extract(sqrt_one_minus_alphas_cumprod, t, x.shape)
    sqrt_recip_alphas_t = extract(sqrt_recip_alphas, t, x.shape)

    model_mean = sqrt_recip_alphas_t*(
        x - betas_t * model(x, t) / sqrt_one_minus_alphas_cumprod_t
    )
    if t_index == 0:
        return model_mean
    else:
        posterior_variance_t = extract(posterior_variance, t, x.shape)
        noise = torch.randn_like(x)
        return model_mean + torch.sqrt(posterior_variance_t)* noise

@torch.no_grad()
def p_sample_loop(model, shape):
    img = torch.randn(shape, device = DEVICE)
    imgs = []

    for i in tqdm(reversed(range(0, TIMESTEPS)), desc = "Sampling loop", total = TIMESTEPS):
        t = torch.full((shape[0],), i, device = DEVICE, dtype = torch.long)
        img = p_sample(model, img, t, i)

    return img

In [ ]:

def get_loss(model, x_start, t):
    noise = torch.randn_like(x_start)

    x_noisy = q_sample(
        x_start=x_start,
        t=t,
        noise=noise
    )

    predicted_noise = model(x_noisy, t)
    loss = F.l1_loss(predicted_noise, noise)

    return loss


def train():
    dataloader = get_data_loader()

    model = SimpleUnet().to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(dataloader, leave=True)

        for images, _ in loop:
            images = images.to(DEVICE)

            t = torch.randint(
                0,
                TIMESTEPS,
                (images.shape[0],),
                device=DEVICE
            ).long()

            loss = get_loss(model, images, t)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loop.set_description(
                f"Epoch [{epoch + 1}/{EPOCHS}]"
            )
            loop.set_postfix(loss=loss.item())

        print(
            f"Epoch {epoch + 1} completed. "
            "Generating sample images..."
        )

        model.eval()

        with torch.no_grad():
            num_images_to_sample = 16
            sample_shape = (
                num_images_to_sample,
                1,
                IMG_SIZE,
                IMG_SIZE
            )

            generated_images = p_sample_loop(
                model, sample_shape
            )

            generated_images = (
                generated_images.clamp(-1, 1) + 1
            ) * 0.5

            save_image(
                generated_images,
                os.path.join(
                    OUTPUT_DIR,
                    f"epoch_{epoch + 1}_sample.png"
                ),
                nrow=4
            )

        print(
            f"Sample images saved for epoch {epoch + 1}."
        )

    torch.save(
        model.state_dict(),
        os.path.join(OUTPUT_DIR, "diffusion_mnist.pth")
    )

    print("Training finished and model saved.")


if __name__ == "__main__":
    print("Starting Diffusion Model Training...")
    print(f"Device: {DEVICE}")
    print(f"Timesteps: {TIMESTEPS}")
    print(f"Batch Size: {BATCH_SIZE}")

    train()

    print(
        "All steps complete. Check the diffusion_outputs directory."
    )
